# Unsupervised Learning with Autoencoders — Cats vs Dogs
## No Labels Needed!

### What is an Autoencoder?
An autoencoder learns to **compress** images into a small latent representation and then **reconstruct** them — without using any labels.

```
Input Image (128×128) → ENCODER → Latent Space (small vector) → DECODER → Reconstructed Image (128×128)
```

### Why unsupervised?
- In supervised learning (v1): we told the model "this is a cat, this is a dog"
- In unsupervised learning (this notebook): model discovers patterns BY ITSELF
- No labels are used during training — only the images themselves

### What we'll do:
1. Build a Convolutional Autoencoder (compresses & reconstructs images)
2. Train it to reconstruct cat/dog images
3. Visualize what the model learned in the latent space
4. Use K-Means clustering on latent vectors (does it separate cats from dogs?)
5. Anomaly detection (find unusual images)

In [0]:
%pip install torch torchvision scikit-learn

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import optim
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score
import os
import subprocess

In [0]:
# ============================================================
# DATASET SETUP — Same dataset as v1 (Cats & Dogs)
# Key difference: We DON'T use labels for training!
# ============================================================

data_dir = "/tmp/Cat_Dog_data"

# Download if not present
if not os.path.exists(data_dir):
    subprocess.run(["wget", "-q", "http://cdn.iiith.talentsprint.com/aiml/Experiment_related_data/Cat_Dog_data_B17.zip", "-O", "/tmp/Cat_Dog_data_B17.zip"], check=True)
    subprocess.run(["unzip", "-qq", "/tmp/Cat_Dog_data_B17.zip", "-d", "/tmp/"], check=True)
    print("Dataset downloaded successfully!")
else:
    print("Dataset already exists.")

# Transforms — same as v1 (Resize, Grayscale, ToTensor, Normalize)
image_size = (128, 128)
transformations = transforms.Compose([
    transforms.Resize(image_size),
    transforms.Grayscale(num_output_channels=1),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# Load dataset
batch_size = 64
train_set = datasets.ImageFolder(f'{data_dir}/train', transform=transformations)
test_set = datasets.ImageFolder(f'{data_dir}/test', transform=transformations)

train_loader = torch.utils.data.DataLoader(train_set, batch_size=batch_size, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_set, batch_size=batch_size, shuffle=False)

class_names = train_set.classes
print(f"Training images: {len(train_set)}")
print(f"Test images: {len(test_set)}")
print(f"Classes: {class_names}")
print(f"\nNote: Labels exist in dataset but we WON'T use them for training!")
print(f"We'll only use them AFTER training to evaluate if the model discovered the groups.")

## The Convolutional Autoencoder Architecture

```
ENCODER (compresses):                    DECODER (reconstructs):
─────────────────────                    ────────────────────────
Input (1, 128, 128)                      Latent vector (256,)
    ↓ Conv(1→16) + ReLU + MaxPool            ↓ Linear(256→1024)
(16, 64, 64)                             (16, 8, 8)
    ↓ Conv(16→32) + ReLU + MaxPool           ↓ ConvTranspose(16→16) + ReLU
(32, 32, 32)                             (16, 16, 16)
    ↓ Conv(32→64) + ReLU + MaxPool           ↓ ConvTranspose(16→32) + ReLU
(64, 16, 16)                             (32, 32, 32)
    ↓ Conv(64→16) + ReLU + MaxPool           ↓ ConvTranspose(32→16) + ReLU
(16, 8, 8)                               (16, 64, 64)
    ↓ Flatten + Linear                       ↓ ConvTranspose(16→1) + Tanh
(256,) ← LATENT SPACE                   (1, 128, 128) ← RECONSTRUCTED IMAGE
```

**Loss function:** MSELoss (Mean Squared Error) — how different is the reconstruction from the original?

**No labels needed:** The model trains by comparing its output to its INPUT (not to a class label).

In [0]:
# ============================================================
# CONVOLUTIONAL AUTOENCODER
# Encoder: compresses 128×128 image → 256-dim latent vector
# Decoder: reconstructs 256-dim latent vector → 128×128 image
# ============================================================

class ConvAutoencoder(nn.Module):
    def __init__(self, latent_dim=256):
        super(ConvAutoencoder, self).__init__()
        self.latent_dim = latent_dim
        
        # ===== ENCODER =====
        # Compresses image into a small latent vector
        self.encoder_conv = nn.Sequential(
            # Layer 1: (1, 128, 128) → (16, 64, 64)
            nn.Conv2d(1, 16, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            
            # Layer 2: (16, 64, 64) → (32, 32, 32)
            nn.Conv2d(16, 32, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            
            # Layer 3: (32, 32, 32) → (64, 16, 16)
            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            
            # Layer 4: (64, 16, 16) → (16, 8, 8)
            nn.Conv2d(64, 16, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
        )
        
        # Flatten + compress to latent vector
        self.encoder_fc = nn.Linear(16 * 8 * 8, latent_dim)
        
        # ===== DECODER =====
        # Reconstructs image from latent vector
        self.decoder_fc = nn.Linear(latent_dim, 16 * 8 * 8)
        
        self.decoder_conv = nn.Sequential(
            # (16, 8, 8) → (16, 16, 16)
            nn.ConvTranspose2d(16, 16, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            
            # (16, 16, 16) → (32, 32, 32)
            nn.ConvTranspose2d(16, 32, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            
            # (32, 32, 32) → (16, 64, 64)
            nn.ConvTranspose2d(32, 16, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            
            # (16, 64, 64) → (1, 128, 128)
            nn.ConvTranspose2d(16, 1, kernel_size=4, stride=2, padding=1),
            nn.Tanh()  # Output range [-1, 1] (matches our normalization)
        )
    
    def encode(self, x):
        """Compress image to latent vector."""
        x = self.encoder_conv(x)
        x = x.view(x.size(0), -1)  # Flatten
        z = self.encoder_fc(x)      # Latent vector
        return z
    
    def decode(self, z):
        """Reconstruct image from latent vector."""
        x = self.decoder_fc(z)
        x = x.view(x.size(0), 16, 8, 8)  # Reshape to feature maps
        x = self.decoder_conv(x)
        return x
    
    def forward(self, x):
        """Full forward pass: encode then decode."""
        z = self.encode(x)
        reconstructed = self.decode(z)
        return reconstructed, z

# Create model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
autoencoder = ConvAutoencoder(latent_dim=256).to(device)

print(f"Device: {device}")
print(f"Latent dimension: 256 (compresses 128×128=16,384 pixels into 256 numbers)")
print(f"Compression ratio: {128*128/256:.0f}x")
print(f"\nTotal parameters: {sum(p.numel() for p in autoencoder.parameters()):,}")
print(f"\nModel summary:")
print(autoencoder)

In [0]:
# ============================================================
# TRAINING — UNSUPERVISED (No labels used!)
# Loss = MSE between original image and reconstructed image
# The model learns to compress & reconstruct without knowing cat/dog
# ============================================================

# Loss function: how different is reconstruction from original?
criterion = nn.MSELoss()

# Optimizer
optimizer = optim.Adam(autoencoder.parameters(), lr=0.001)

# Training
num_epochs = 15
train_losses = []

print("Training Autoencoder (UNSUPERVISED — no labels used!)")
print("="*55)

for epoch in range(num_epochs):
    autoencoder.train()
    running_loss = 0.0
    
    for images, _ in train_loader:  # ← Notice: we IGNORE labels (_)
        images = images.to(device)
        
        # Forward pass
        reconstructed, latent = autoencoder(images)
        
        # Loss: compare reconstruction to ORIGINAL (not to a label!)
        loss = criterion(reconstructed, images)
        
        # Backward + optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
    
    epoch_loss = running_loss / len(train_loader)
    train_losses.append(epoch_loss)
    print(f'Epoch [{epoch+1}/{num_epochs}], Reconstruction Loss: {epoch_loss:.6f}')

print(f'\nTraining complete! Final loss: {train_losses[-1]:.6f}')
print(f'Lower loss = better reconstruction = model learned good features')

In [0]:
# ============================================================
# VISUALIZE RECONSTRUCTIONS
# Top row: Original images
# Bottom row: What the autoencoder reconstructed
# ============================================================

autoencoder.eval()

# Get a batch of test images
test_images, test_labels = next(iter(test_loader))
test_images = test_images.to(device)

with torch.no_grad():
    reconstructed, latent = autoencoder(test_images)

# Plot originals vs reconstructions
fig, axes = plt.subplots(3, 8, figsize=(18, 7))

for i in range(8):
    # Original
    axes[0, i].imshow(test_images[i].cpu().squeeze(), cmap='gray')
    axes[0, i].set_title(f'{class_names[test_labels[i]]}', fontsize=9)
    axes[0, i].axis('off')
    
    # Reconstructed
    axes[1, i].imshow(reconstructed[i].cpu().squeeze(), cmap='gray')
    axes[1, i].set_title('Reconstructed', fontsize=9)
    axes[1, i].axis('off')
    
    # Difference (error map)
    diff = torch.abs(test_images[i] - reconstructed[i]).cpu().squeeze()
    axes[2, i].imshow(diff, cmap='hot')
    axes[2, i].set_title('Error', fontsize=9)
    axes[2, i].axis('off')

axes[0, 0].set_ylabel('Original', fontsize=12)
axes[1, 0].set_ylabel('Reconstructed', fontsize=12)
axes[2, 0].set_ylabel('Difference', fontsize=12)
plt.suptitle('Autoencoder: Original vs Reconstructed (Unsupervised)', fontsize=14)
plt.tight_layout()
plt.show()

# Training loss curve
plt.figure(figsize=(10, 4))
plt.plot(range(1, num_epochs+1), train_losses, 'b-o', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Reconstruction Loss (MSE)')
plt.title('Autoencoder Training Loss')
plt.grid(True, alpha=0.3)
plt.show()

In [0]:
# ============================================================
# LATENT SPACE ANALYSIS — Can the model separate cats from dogs?
# We extract latent vectors for ALL test images, then:
# 1. Apply K-Means (k=2) — does it find 2 clusters?
# 2. Compare clusters to actual labels — did it discover cat/dog?
# 3. Visualize with PCA (reduce 256 dims → 2 dims for plotting)
# ============================================================

autoencoder.eval()

all_latents = []
all_labels = []

# Extract latent vectors for all test images
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        _, latent = autoencoder(images)
        all_latents.append(latent.cpu().numpy())
        all_labels.extend(labels.numpy())

all_latents = np.concatenate(all_latents, axis=0)
all_labels = np.array(all_labels)

print(f"Latent vectors shape: {all_latents.shape}")
print(f"Each image compressed to a {all_latents.shape[1]}-dimensional vector\n")

# --- K-Means Clustering (k=2) ---
kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(all_latents)

# Compare clusters to actual labels
ari_score = adjusted_rand_score(all_labels, cluster_labels)
print(f"Adjusted Rand Index (ARI): {ari_score:.4f}")
print(f"  ARI = 1.0: perfect clustering (exactly matches cat/dog labels)")
print(f"  ARI = 0.0: random clustering (no better than chance)")
print(f"  ARI > 0.5: good separation discovered!\n")

# Check cluster-label mapping
cluster_0_cats = np.sum((cluster_labels == 0) & (all_labels == 0))
cluster_0_dogs = np.sum((cluster_labels == 0) & (all_labels == 1))
cluster_1_cats = np.sum((cluster_labels == 1) & (all_labels == 0))
cluster_1_dogs = np.sum((cluster_labels == 1) & (all_labels == 1))

print("Cluster composition:")
print(f"  Cluster 0: {cluster_0_cats} cats, {cluster_0_dogs} dogs")
print(f"  Cluster 1: {cluster_1_cats} cats, {cluster_1_dogs} dogs")

# --- PCA Visualization (256D → 2D) ---
pca = PCA(n_components=2)
latents_2d = pca.fit_transform(all_latents)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Colored by ACTUAL labels (cat=blue, dog=red)
scatter1 = ax1.scatter(latents_2d[:, 0], latents_2d[:, 1], 
                       c=all_labels, cmap='coolwarm', alpha=0.5, s=10)
ax1.set_title('Latent Space — Colored by ACTUAL Labels', fontsize=12)
ax1.set_xlabel('PCA Component 1')
ax1.set_ylabel('PCA Component 2')
ax1.legend(handles=scatter1.legend_elements()[0], labels=class_names)

# Plot 2: Colored by K-MEANS clusters
scatter2 = ax2.scatter(latents_2d[:, 0], latents_2d[:, 1], 
                       c=cluster_labels, cmap='viridis', alpha=0.5, s=10)
ax2.set_title('Latent Space — Colored by K-Means Clusters', fontsize=12)
ax2.set_xlabel('PCA Component 1')
ax2.set_ylabel('PCA Component 2')
ax2.legend(handles=scatter2.legend_elements()[0], labels=['Cluster 0', 'Cluster 1'])

plt.suptitle('Does the Autoencoder Discover Cats vs Dogs Without Labels?', fontsize=14)
plt.tight_layout()
plt.show()

print(f"\nInterpretation:")
print(f"If the two plots look similar → model discovered cat/dog separation on its own!")
print(f"If very different → model learned OTHER features (texture, brightness, pose)")

In [0]:
# ============================================================
# ANOMALY DETECTION — Find unusual/hard-to-reconstruct images
# Idea: images with HIGH reconstruction error are "anomalies"
# These might be blurry, unusual poses, or ambiguous images
# ============================================================

autoencoder.eval()
reconstruction_errors = []
all_test_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        reconstructed, _ = autoencoder(images)
        
        # Per-image MSE error
        errors = ((images - reconstructed) ** 2).mean(dim=[1, 2, 3])  # average per image
        reconstruction_errors.extend(errors.cpu().numpy())
        all_test_labels.extend(labels.numpy())

reconstruction_errors = np.array(reconstruction_errors)
all_test_labels = np.array(all_test_labels)

# --- Plot error distribution ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of errors
ax1.hist(reconstruction_errors[all_test_labels == 0], bins=30, alpha=0.6, label='Cats', color='blue')
ax1.hist(reconstruction_errors[all_test_labels == 1], bins=30, alpha=0.6, label='Dogs', color='red')
ax1.axvline(x=np.percentile(reconstruction_errors, 95), color='black', linestyle='--', label='95th percentile')
ax1.set_xlabel('Reconstruction Error')
ax1.set_ylabel('Count')
ax1.set_title('Reconstruction Error Distribution')
ax1.legend()

# Sorted errors
ax2.plot(sorted(reconstruction_errors), 'b-', linewidth=1)
ax2.axhline(y=np.percentile(reconstruction_errors, 95), color='r', linestyle='--', label='Anomaly threshold (95%)')
ax2.set_xlabel('Image Index (sorted)')
ax2.set_ylabel('Reconstruction Error')
ax2.set_title('Sorted Reconstruction Errors')
ax2.legend()

plt.tight_layout()
plt.show()

# --- Show top anomalies (hardest to reconstruct) ---
top_anomaly_idx = np.argsort(reconstruction_errors)[-8:]  # top 8 worst

fig, axes = plt.subplots(2, 8, figsize=(18, 5))
for i, idx in enumerate(top_anomaly_idx):
    img, label = test_set[idx]
    axes[0, i].imshow(img.squeeze(), cmap='gray')
    axes[0, i].set_title(f'{class_names[label]}\nerr={reconstruction_errors[idx]:.4f}', fontsize=8)
    axes[0, i].axis('off')
    
    # Show reconstruction
    with torch.no_grad():
        recon, _ = autoencoder(img.unsqueeze(0).to(device))
    axes[1, i].imshow(recon.cpu().squeeze(), cmap='gray')
    axes[1, i].set_title('Reconstructed', fontsize=8)
    axes[1, i].axis('off')

axes[0, 0].set_ylabel('Original', fontsize=11)
axes[1, 0].set_ylabel('Reconstructed', fontsize=11)
plt.suptitle('Top Anomalies — Hardest to Reconstruct (unusual images)', fontsize=13)
plt.tight_layout()
plt.show()

print(f"\nAnomaly Detection Summary:")
print(f"  Mean error (cats):  {reconstruction_errors[all_test_labels==0].mean():.6f}")
print(f"  Mean error (dogs):  {reconstruction_errors[all_test_labels==1].mean():.6f}")
print(f"  Anomaly threshold (95th percentile): {np.percentile(reconstruction_errors, 95):.6f}")
print(f"  Images above threshold: {np.sum(reconstruction_errors > np.percentile(reconstruction_errors, 95))}")

## Summary: Supervised (v1) vs Unsupervised (v2)

| | Supervised (CNN - v1) | Unsupervised (Autoencoder - v2) |
| --- | --- | --- |
| **Labels used?** | Yes (cat=0, dog=1) | No |
| **Goal** | Predict class | Learn data representation |
| **Loss function** | NLLLoss (prediction vs label) | MSELoss (reconstruction vs original) |
| **Output** | Class probability | Reconstructed image + latent vector |
| **What it learns** | "What makes a cat vs dog" | "What makes these images look like themselves" |
| **Evaluation** | Accuracy (%) | Reconstruction quality + clustering ARI |
| **Use cases** | Classification | Compression, anomaly detection, clustering, generation |

### Key Insight:
- **Supervised**: You TELL the model the answer → it learns a decision boundary
- **Unsupervised**: Model DISCOVERS structure on its own → finds natural groupings

The autoencoder doesn't know "cat" or "dog" — it just knows "these images look similar" and groups them in latent space.